In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="langchain_text_splitters")

In [3]:
from langchain_chroma import Chroma

# from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv

In [5]:
# Define the persistent directory for Chroma database
persistent_directory = "db/chroma_db"

# Load embeddings and vector store
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2480.64it/s]


In [6]:
# Query to test
query = "How much did Microsoft pay to acquire GitHub?"
# query = "How do you plant tomatoes in a garden?"
print(f"Query: {query}\n")

# ──────────────────────────────────────────────────────────────────
# METHOD 1: Basic Similarity Search
# Returns the top k most similar documents
# ──────────────────────────────────────────────────────────────────

print("=== METHOD 1: Similarity Search (k=3) ===")
retriever = db.as_retriever(search_kwargs={"k": 3})

docs = retriever.invoke(query)
print(f"Retrieved {len(docs)} documents:\n")

for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(f"{doc.page_content}\n")

print("-" * 60)

Query: How much did Microsoft pay to acquire GitHub?

=== METHOD 1: Similarity Search (k=3) ===
Retrieved 3 documents:

Document 1:
117. "Microsoft's 2018, part 1: Open source, wobbly Windows and everyone's going to the cloud"
(https://www.theregister.co.uk/2018/12/25/microsoft_year_in_review_2018/). The Register.
Archived (https://web.archive.org/web/20190103060059/https://www.theregister.co.uk/2018/
12/25/microsoft_year_in_review_2018/) from the original on January 3, 2019. Retrieved
January 3, 2019.

118. "Microsoft to acquire GitHub for $7.5 billion" (https://news.microsoft.com/2018/06/04/microso
ft-to-acquire-github-for-7-5-billion/). Microsoft. June 4, 2018. Archived (https://web.archive.or
g/web/20180604142244/https://news.microsoft.com/2018/06/04/microsoft-to-acquire-github-
for-7-5-billion/) from the original on June 4, 2018.

Document 2:
119. "Microsoft completes GitHub acquisition" (https://web.archive.org/web/20190112212059/http
s://www.msn.com/en-us/news/technology/microso

In [8]:
# ──────────────────────────────────────────────────────────────────
# METHOD 2: Similarity with Score Threshold
# Only returns documents above a certain similarity score
# ──────────────────────────────────────────────────────────────────

print("\n=== METHOD 2: Similarity with Score Threshold ===")
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 3,
        "score_threshold": 0.3  # Only return docs with similarity >= 0.3
    }
)

docs = retriever.invoke(query)
print(f"Retrieved {len(docs)} documents (threshold: 0.3):\n")

for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(f"{doc.page_content}\n")



=== METHOD 2: Similarity with Score Threshold ===
Retrieved 3 documents (threshold: 0.3):

Document 1:
117. "Microsoft's 2018, part 1: Open source, wobbly Windows and everyone's going to the cloud"
(https://www.theregister.co.uk/2018/12/25/microsoft_year_in_review_2018/). The Register.
Archived (https://web.archive.org/web/20190103060059/https://www.theregister.co.uk/2018/
12/25/microsoft_year_in_review_2018/) from the original on January 3, 2019. Retrieved
January 3, 2019.

118. "Microsoft to acquire GitHub for $7.5 billion" (https://news.microsoft.com/2018/06/04/microso
ft-to-acquire-github-for-7-5-billion/). Microsoft. June 4, 2018. Archived (https://web.archive.or
g/web/20180604142244/https://news.microsoft.com/2018/06/04/microsoft-to-acquire-github-
for-7-5-billion/) from the original on June 4, 2018.

Document 2:
119. "Microsoft completes GitHub acquisition" (https://web.archive.org/web/20190112212059/http
s://www.msn.com/en-us/news/technology/microsoft-completes-github-acquisit

In [9]:
# ──────────────────────────────────────────────────────────────────
# METHOD 3: Maximum Marginal Relevance (MMR)
# Balances relevance and diversity - avoids redundant results
# ──────────────────────────────────────────────────────────────────

print("\n=== METHOD 3: Maximum Marginal Relevance (MMR) ===")
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,           # Final number of docs
        "fetch_k": 10,    # Initial pool to select from
        "lambda_mult": 0.5  # 0=max diversity, 1=max relevance
    }
)

docs = retriever.invoke(query)
print(f"Retrieved {len(docs)} documents (λ=0.5):\n")

for i, doc in enumerate(docs, 1):
    print(f"Document {i}:")
    print(f"{doc.page_content}\n")

print("=" * 60)
print("Done! Try different queries or parameters to see the differences.")


=== METHOD 3: Maximum Marginal Relevance (MMR) ===
Retrieved 3 documents (λ=0.5):

Document 1:
117. "Microsoft's 2018, part 1: Open source, wobbly Windows and everyone's going to the cloud"
(https://www.theregister.co.uk/2018/12/25/microsoft_year_in_review_2018/). The Register.
Archived (https://web.archive.org/web/20190103060059/https://www.theregister.co.uk/2018/
12/25/microsoft_year_in_review_2018/) from the original on January 3, 2019. Retrieved
January 3, 2019.

118. "Microsoft to acquire GitHub for $7.5 billion" (https://news.microsoft.com/2018/06/04/microso
ft-to-acquire-github-for-7-5-billion/). Microsoft. June 4, 2018. Archived (https://web.archive.or
g/web/20180604142244/https://news.microsoft.com/2018/06/04/microsoft-to-acquire-github-
for-7-5-billion/) from the original on June 4, 2018.

Document 2:
In June 2016, Microsoft announced a project named Microsoft Azure Information Protection. It aims to
help enterprises protect their data as it moves between servers and devices